# AlphaEarth-System — Linhe PEFT Training on Colab

Trains Prithvi + PEFT on the Linhe RGB patch subset packaged by
`scripts/package_colab_data.py`. Saves checkpoints back to Google Drive so a
mid-session Colab kill can resume from the last epoch.

**Before running:**
1. Upload `linhe_<quarters>.tar.gz` + its `.sha256` to Drive (MyDrive root).
2. Runtime → Change runtime type → L4 GPU (or T4 for smoke).
3. Run cells top-to-bottom.

In [ ]:
# 1. Mount Drive, pick archives
from google.colab import drive
drive.mount('/content/drive')

# Two-half packaging keeps each tar.gz under 500 MB so Drive 1 GB free tier
# can hold both simultaneously. Order does not matter — the second tar's
# _index.parquet replaces the first; we merge below in cell 5.
ARCHIVES = [
    'linhe_2025_h1.tar.gz',   # Q1 + Q2  (~420 MB)
    'linhe_2025_h2.tar.gz',   # Q3 + Q4  (~408 MB)
]
CHECKPOINT_DIR = '/content/drive/MyDrive/linhe_ckpt'
RESULTS_DIR = '/content/drive/MyDrive/linhe_results'

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Drive mounted; checkpoint dir:', CHECKPOINT_DIR)

In [ ]:
# 2. GPU check
!nvidia-smi | head -20

In [ ]:
# 3. Clone repo (or pull if already cloned)
%cd /content
REPO_URL = 'https://github.com/zhouning/alphaearth-training-system.git'
REPO_DIR = '/content/alphaearth-training-system'
import os, subprocess
if os.path.isdir(REPO_DIR + '/.git'):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
%cd $REPO_DIR
!git log --oneline -5

In [ ]:
# 4. Install deps
!pip install -q -e .
!pip install -q pyarrow scikit-learn matplotlib

In [ ]:
# 5. Copy + verify + extract each archive, then merge indexes
import shutil, subprocess, os
import pandas as pd

merged_idx_rows, merged_osm_rows = [], []
for arc in ARCHIVES:
    src = f'/content/drive/MyDrive/{arc}'
    sha_src = src + '.sha256'
    dst = f'/content/{arc}'
    assert os.path.exists(src), f'Upload {arc} to Drive first'
    shutil.copy(src, dst)
    shutil.copy(sha_src, '/content/')
    print(f'verify {arc}')
    # Strip any trailing  in the sha256 file (Windows-written checksums break sha256sum -c on Linux).
    subprocess.run(['sed', '-i', 's/\r$//', f'/content/{os.path.basename(sha_src)}'], check=True)
    subprocess.run(['sha256sum', '-c', f'/content/{os.path.basename(sha_src)}'],
                   cwd='/content', check=True)
    print(f'extract {arc}')
    subprocess.run(['tar', 'xzf', dst, '-C', REPO_DIR + '/'], check=True)
    # snapshot per-archive index before next archive overwrites it
    merged_idx_rows.append(pd.read_parquet('data/linhe_patches/_index.parquet'))
    merged_osm_rows.append(pd.read_parquet('data/linhe_patches/_osm_index.parquet'))

merged_idx = pd.concat(merged_idx_rows, ignore_index=True).drop_duplicates('patch_path')
merged_osm = pd.concat(merged_osm_rows, ignore_index=True).drop_duplicates('patch_path')
merged_idx.to_parquet('data/linhe_patches/_index.parquet')
merged_osm.to_parquet('data/linhe_patches/_osm_index.parquet')
print(f'merged: {len(merged_idx)} patches, {merged_idx["scene_id"].nunique()} scenes')
print(merged_idx['quarter'].value_counts().to_dict())

In [ ]:
# 6. Download Prithvi weights from HuggingFace (350 MB)
import os
os.makedirs('data/weights/prithvi', exist_ok=True)
if not os.path.exists('data/weights/prithvi/Prithvi_100M.pt'):
    !wget -q https://huggingface.co/ibm-nasa-geospatial/Prithvi-100M/resolve/main/Prithvi_100M.pt -O data/weights/prithvi/Prithvi_100M.pt
!ls -la data/weights/prithvi/

In [ ]:
# 7. GPU throughput sanity check (~30 s)
!python scripts/bench_prithvi_throughput.py --steps 20 --batch-size 16

In [ ]:
# 8. Run 1.a buildings benchmark (synth baseline)
CONFIG_1A = 'geoadapter/bench/configs/linhe_buildings.yaml'
OUTPUT_1A = f'{RESULTS_DIR}/linhe_buildings.json'
!python -m geoadapter.bench.run_benchmark \
    --config {CONFIG_1A} \
    --output {OUTPUT_1A} \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --checkpoint-every 2

In [ ]:
# 8b. Extract LULC masks (1.b prerequisite)
# Upload linhe_lulc_masks.tar.gz (37.7 MB) to Drive root first.
import shutil, os
LULC_ARC = 'linhe_lulc_masks.tar.gz'
lulc_src = f'/content/drive/MyDrive/{LULC_ARC}'
if os.path.exists(lulc_src):
    shutil.copy(lulc_src, f'/content/{LULC_ARC}')
    !tar xzf /content/{LULC_ARC} -C data/linhe_patches/
    print('LULC masks extracted')
    import pandas as pd
    lulc_idx = pd.read_parquet('data/linhe_patches/_lulc_index.parquet')
    # Fix Windows backslash paths -> Linux forward slash
    for col in ['patch_path', 'lulc_path']:
        if col in lulc_idx.columns:
            lulc_idx[col] = lulc_idx[col].str.replace('\\\\', '/', regex=False).str.replace('\\', '/', regex=False)
    lulc_idx.to_parquet('data/linhe_patches/_lulc_index.parquet')
    print(f'LULC index: {len(lulc_idx)} rows, years={sorted(lulc_idx["year"].unique())}')
    print(f'Sample patch_path: {lulc_idx.iloc[0]["patch_path"]}')
    # Sanity check: file exists
    sample_p = lulc_idx.iloc[0]["patch_path"]
    print(f'File exists: {os.path.exists(sample_p)}')
else:
    print(f'WARNING: {lulc_src} not found on Drive — skip 1.b training')

In [ ]:
# 8c. Run 1.b LULC 6-class segmentation benchmark
# Use separate checkpoint dir to avoid shape mismatch with 1.a's 2-class head.
CONFIG_1B = 'geoadapter/bench/configs/linhe_lulc.yaml'
OUTPUT_1B = f'{RESULTS_DIR}/linhe_lulc_seg.json'
CHECKPOINT_DIR_1B = f'{CHECKPOINT_DIR}_lulc'
import os
os.makedirs(CHECKPOINT_DIR_1B, exist_ok=True)
!python -m geoadapter.bench.run_benchmark \
    --config {CONFIG_1B} \
    --output {OUTPUT_1B} \
    --checkpoint-dir {CHECKPOINT_DIR_1B} \
    --checkpoint-every 2

In [ ]:
# 9. Peek results (1.a + 1.b)
import json, os
import pandas as pd

all_results = []
for label, path in [('1.a buildings', OUTPUT_1A), ('1.b LULC', OUTPUT_1B)]:
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        for r in data:
            r['task'] = label
        all_results.extend(data)

if all_results:
    df = pd.DataFrame(all_results)
    print(f'Columns: {list(df.columns)}')
    # Pick metric column dynamically
    metric_col = 'mIoU' if 'mIoU' in df.columns else ('overall_accuracy' if 'overall_accuracy' in df.columns else None)
    show_cols = ['task', 'method', 'seed', 'trainable_params']
    if metric_col:
        show_cols.append(metric_col)
    show_cols = [c for c in show_cols if c in df.columns]
    print(df[show_cols].sort_values(show_cols[:2], ascending=[True, False]).to_string())
else:
    print('No results yet — run cell 8 and/or 8c first')

## Usage notes

- **Out-of-time kill**: Colab may cut the session after ~24 h / idle. Re-running
  cell 8/8c resumes from the last checkpoint + skips completed experiments.
- **Budget**: 1.a (2 methods x 3 seeds x 30 epochs) + 1.b (same) ~ 50 L4
  GPU-hours ~ 250 compute units (user has 600/month).
- **1.b only**: If 1.a is already done, skip cell 8 and run 8b + 8c directly.
- **LULC tar.gz**: `linhe_lulc_masks.tar.gz` (37.7 MB) contains 107760 mask files
  for 2022+2023. Extract into `data/linhe_patches/` alongside RGB patches.
- **Checkpoint cleanup**: after a successful full run, delete
  `/content/drive/MyDrive/linhe_ckpt/` to free Drive space.